# Lección 08 - Patrón de Diseño Multi-Agente


## Configuración


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ¿Por qué Sistemas Multi-Agente?

Las tareas del mundo real como la planificación de viajes involucran muchos tipos diferentes de experiencia: logística, conocimiento local, presupuesto y más. Un solo agente que intenta manejar todo se vuelve rápidamente inmanejable.

Los sistemas multi-agente resuelven esto mediante la **especialización**: cada agente se enfoca en un área de experiencia, produciendo resultados de mayor calidad que un generalista. También mejoran la **escalabilidad**: puedes agregar nuevos agentes (por ejemplo, un especialista en vuelos, un crítico de restaurantes) sin reescribir el flujo de trabajo existente. Los agentes se componen a través de una cadena estructurada, pasando el contexto de uno a otro.


## Creación de Agentes Especializados


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Construcción de un flujo de trabajo secuencial

`WorkflowBuilder` te permite conectar agentes en un grafo dirigido. Aquí creamos una canalización simple de dos pasos: el **TravelPlanner** elabora el itinerario, luego el **TravelConcierge** lo revisa y mejora.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Añadiendo Más Agentes al Flujo de Trabajo

Una de las mayores ventajas del patrón multi-agente es lo fácil que es de extender. A continuación añadimos un agente **BudgetReviewer** que verifica el plan con respecto al presupuesto del viajero, señala los elementos que podrían hacer que los costos superen el límite y sugiere alternativas para ahorrar dinero. El flujo de trabajo ahora ejecuta tres agentes en secuencia:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

En esta lección aprendiste a:

1. **Crear agentes especializados** — cada uno con un rol específico (planificación, conserjería, revisión de presupuesto).  
2. **Conectar agentes en un flujo de trabajo secuencial** usando `WorkflowBuilder` y `add_edge`.  
3. **Transmitir la salida** desde una tubería multiagente, rastreando qué agente está hablando.  
4. **Extender un flujo de trabajo** agregando nuevos agentes a la cadena sin modificar los existentes.  

El patrón de diseño multiagente mantiene a cada agente simple mientras produce resultados más ricos y revisados exhaustivamente que cualquier agente individual podría lograr por sí solo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por un humano. No nos hacemos responsables por malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
